In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

print("1. Загрузка тестовых данных и моделей...")
df_test_raw = pd.read_csv('test_real_estate_samples.csv', encoding='utf-8')
df_original = df_test_raw.copy()

model_sale = joblib.load('best_model_sale.pkl')
scaler_sale = joblib.load('scaler_sale.pkl')
feature_cols = joblib.load('feature_cols.pkl')

print(f"Загружено {len(df_test_raw)} объектов для тестирования.")

print("\n2. Предобработка тестовых данных (полная, как при обучении)...")
df_test = df_test_raw.drop(columns=['furnished', 'end_date', 'id'], errors='ignore')

room_mapping = {'1+0': 0, '1+1': 1, '2+1': 2, '2+2': 3, '3+1': 4, '3+2': 5,
                '4+1': 6, '4+2': 7, '5+1': 8, '5+2': 9, '+': 10}
df_test['room_count_enc'] = df_test['room_count'].map(room_mapping).fillna(10)

def parse_age(x):
    if pd.isna(x): return np.nan
    x = str(x)
    if x == '0': return 0
    if 've üzeri' in x: return int(x.split()[0]) + 5
    if '-' in x:
        parts = x.split('-')
        return (int(parts[0]) + int(parts[1].split()[0])) / 2
    try: return float(x)
    except: return np.nan
df_test['building_age_num'] = df_test['building_age'].apply(parse_age)

def parse_floor(x):
    if pd.isna(x): return np.nan
    x = str(x)
    if 've üzeri' in x: return int(x.split()[0]) + 5
    if 'Yüksek Giriş' in x or 'Zemin' in x: return 0
    try: return float(x)
    except: return np.nan
df_test['floor_no_num'] = df_test['floor_no'].apply(parse_floor)
df_test['total_floor_num'] = df_test['total_floor_count'].apply(parse_floor)

df_test['district'] = df_test['address'].apply(lambda x: x.split('/')[-1] if pd.notna(x) else 'Unknown')
nominal_cols = ['type', 'sub_type', 'heating_type', 'price_currency', 'district']
df_test = pd.get_dummies(df_test, columns=nominal_cols, prefix=nominal_cols, drop_first=False)

central_districts = ['Beşiktaş', 'Şişli', 'Kadıköy', 'Bakırköy', 'Ataşehir', 'Levent', 'Konak']
central_cols = [c for c in df_test.columns if any(d in c for d in central_districts)]
if len(central_cols) > 0:
    df_test['is_central'] = df_test[central_cols].sum(axis=1).clip(upper=1)
else:
    df_test['is_central'] = 0

for col in feature_cols:
    if col not in df_test.columns:
        df_test[col] = 0

# Гарантируем точное совпадение порядка и имен столбцов
X_test = df_test[feature_cols].copy()

print("\n3. Прогнозирование...")
X_test_scaled = scaler_sale.transform(X_test)
# Преобразуем обратно в DataFrame, чтобы удовлетворить требование модели об именах признаков
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_cols)

pred_log = model_sale.predict(X_test_df)
pred_real = np.expm1(pred_log)

df_original['Predicted_Price'] = pred_real

print("\n4. Расчет метрик...")
y_true = df_original['price'].values
y_pred = df_original['Predicted_Price'].values

mae_lib = mean_absolute_error(y_true, y_pred)
rmse_lib = np.sqrt(mean_squared_error(y_true, y_pred))
r2_lib = r2_score(y_true, y_pred)

print(f"MAE (библиотека): {mae_lib:,.2f} TRY")
print(f"RMSE (библиотека): {rmse_lib:,.2f} TRY")
print(f"R2 (библиотека): {r2_lib:.4f}")

mae_manual = np.mean(np.abs(y_true - y_pred))
rmse_manual = np.sqrt(np.mean((y_true - y_pred)**2))
y_mean = np.mean(y_true)
r2_manual = 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - y_mean)**2))

print(f"\nMAE (вручную): {mae_manual:,.2f} TRY")
print(f"RMSE (вручную): {rmse_manual:,.2f} TRY")
print(f"R2 (вручную): {r2_manual:.4f}")

print("\n5. Сохранение результата...")
df_original['ФИО'] = 'Жеребов Даниил Артёмович'
df_original['Группа'] = '23П-2'

output_filename = 'result_real_estate_samples.csv'
df_original.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"Файл успешно сохранен: {output_filename}")

print("\nИтоговая таблица (первые 5 строк):")
display(df_original[['id', 'address', 'room_count', 'size', 'price', 'Predicted_Price', 'ФИО', 'Группа']].head())

1. Загрузка тестовых данных и моделей...
Загружено 20 объектов для тестирования.

2. Предобработка тестовых данных (полная, как при обучении)...

3. Прогнозирование...

4. Расчет метрик...
MAE (библиотека): 419,595.96 TRY
RMSE (библиотека): 608,061.41 TRY
R2 (библиотека): 0.0507

MAE (вручную): 419,595.96 TRY
RMSE (вручную): 608,061.41 TRY
R2 (вручную): 0.0507

5. Сохранение результата...
Файл успешно сохранен: result_real_estate_samples.csv

Итоговая таблица (первые 5 строк):


,id,address,room_count,size,price,Predicted_Price,ФИО,Группа
0,49,İzmir/Aliağa/Yenişakran,1+0,1.0,300000.0,246062.799881,Жеребов Даниил Артёмович,23П-2
1,506,İzmir/Foça/Yeniköy,2+2,250.0,1600000.0,728222.422848,Жеребов Даниил Артёмович,23П-2
2,2,İstanbul/Kartal/Kordonboyu,1+0,43.0,490000.0,215249.700248,Жеребов Даниил Артёмович,23П-2
3,528,Tekirdağ/Çerkezköy/Kızılpınar,3+1,122.0,103600.0,110645.525759,Жеребов Даниил Артёмович,23П-2
4,109,İzmir/Aliağa/Yenişakran,1+0,1.0,300000.0,250663.766914,Жеребов Даниил Артёмович,23П-2
